1. load lib and model

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings

warnings.filterwarnings('ignore')

print("="*70)
print("Part 3: Prediction Demo - Game Success Prediction")
print("="*70)

# load model
model_path = '../Model/best_model.pkl'
feature_columns_path = '../Model/feature_columns.pkl'
model_info_path = '../Model/model_info.pkl'

print("\nLoading saved model...")
best_model = joblib.load(model_path)
feature_columns = joblib.load(feature_columns_path)
model_info = joblib.load(model_info_path)

print(f"Model loaded successfully")
print(f"\nModel Information:")
print(f"  • Model Type: {model_info['model_name']}")
print(f"  • Accuracy: {model_info['accuracy']:.4f} ({model_info['accuracy']*100:.2f}%)")
print(f"  • F1-Score: {model_info['f1_score']:.4f}")
print(f"  • Features: {model_info['n_features']}")

Part 3: Prediction Demo - Game Success Prediction

Loading saved model...
Model loaded successfully

Model Information:
  • Model Type: XGBoost
  • Accuracy: 0.9790 (97.90%)
  • F1-Score: 0.9787
  • Features: 17


2. prepare sample data

In [2]:
print("\n" + "="*70)
print("1. Prepare Sample Data")
print("="*70)

# load full data
df_full = pd.read_csv('../Data/steam_twitch_final_cleaned.csv')

print(f"Full dataset: {df_full.shape}")

# create sample(20 games)
np.random.seed(42)
sample_indices = np.random.choice(df_full.index, size=20, replace=False)
df_sample = df_full.loc[sample_indices].copy()

print(f"\nSample data: {df_sample.shape}")
print(f"Sample includes:")
print(f"  • Actual success: {df_sample['success'].sum()}")
print(f"  • Actual failure: {(df_sample['success']==0).sum()}")

# show sample games
print(f"\nSample games:")
for i, (idx, row) in enumerate(df_sample[['Name', 'success']].iterrows(), 1):
    status = "Success" if row['success'] == 1 else "Failure"
    print(f"  {i:2d}. {row['Name']:<40} {status}")


1. Prepare Sample Data
Full dataset: (1189, 19)

Sample data: (20, 19)
Sample includes:
  • Actual success: 10
  • Actual failure: 10

Sample games:
   1. Chivalry 2                               Failure
   2. Norland                                  Failure
   3. Spelunky 2                               Success
   4. The Survivalists                         Failure
   5. NBA 2K23                                 Failure
   6. Conqueror's Blade                        Failure
   7. NBA 2K17                                 Failure
   8. SnowRunner                               Success
   9. GreedFall                                Failure
  10. Starbound                                Success
  11. FTL: Faster Than Light                   Success
  12. Battle Brothers                          Success
  13. TRIANGLE STRATEGY                        Failure
  14. Nickelodeon All-Star Brawl               Failure
  15. Football Manager 2021                    Success
  16. The Elder Scrolls V

3. predict

In [3]:
print("\n" + "="*70)
print("2. Make Predictions")
print("="*70)

# prepare features
X_sample = df_sample[feature_columns]

# prediction
predictions = best_model.predict(X_sample)
prediction_proba = best_model.predict_proba(X_sample)

# add result
df_sample['predicted_success'] = predictions
df_sample['success_probability'] = prediction_proba[:, 1]

print(f"Predictions complete for {len(df_sample)} games")

# show result
print(f"\nPrediction Results:")
print(f"{'Game Name':<40} | {'Actual':<8} | {'Predicted':<10} | {'Prob':<8} | {'Result'}")
print("-" * 90)

for idx, row in df_sample.iterrows():
    actual = "Success" if row['success'] == 1 else "Failure"
    predicted = "Success" if row['predicted_success'] == 1 else "Failure"
    prob = row['success_probability']

    if row['success'] == row['predicted_success']:
        result = "Correct"
    else:
        result = "Wrong"

    print(f"{row['Name']:<40} | {actual:<8} | {predicted:<10} | {prob:.3f}   | {result}")

# calculate predict success rate
correct = (df_sample['success'] == df_sample['predicted_success']).sum()
accuracy = correct / len(df_sample)

print(f"\nSample Prediction Accuracy: {correct}/{len(df_sample)} = {accuracy*100:.1f}%")


2. Make Predictions
Predictions complete for 20 games

Prediction Results:
Game Name                                | Actual   | Predicted  | Prob     | Result
------------------------------------------------------------------------------------------
Chivalry 2                               | Failure  | Failure    | 0.001   | Correct
Norland                                  | Failure  | Failure    | 0.001   | Correct
Spelunky 2                               | Success  | Success    | 0.999   | Correct
The Survivalists                         | Failure  | Failure    | 0.001   | Correct
NBA 2K23                                 | Failure  | Failure    | 0.001   | Correct
Conqueror's Blade                        | Failure  | Failure    | 0.001   | Correct
NBA 2K17                                 | Failure  | Failure    | 0.001   | Correct
SnowRunner                               | Success  | Success    | 0.999   | Correct
GreedFall                                | Failure  | Failure    | 0

4.key observations

In [17]:


print("\n" + "="*70)
print("What Makes a Game Succeed?")
print("="*70)

# Feature importance
if hasattr(best_model, 'feature_importances_'):
    feature_imp = pd.DataFrame({
        'Feature': feature_columns,
        'Importance': best_model.feature_importances_
    }).sort_values('Importance', ascending=False)

    # Business explanations for ALL features
    explanations = {
        'positive_ratio': 'Player satisfaction - aim for 85%+ positive reviews',
        'total_reviews': 'Player engagement - more reviews = more visibility',
        'Recommendations': 'Word-of-mouth - players actively recommend your game',
        'Hours_watched': 'Marketing reach - Twitch/streaming visibility',
        'Hours_streamed': 'Streaming activity - how much streamers play it',
        'Avg_viewers': 'Streaming appeal - average viewer count',
        'Peak_viewers': 'Viral potential - maximum viewer count',
        'Streamers': 'Streamer interest - how many people stream it',
        'watch_per_stream_hour': 'Viewer efficiency - watch time per stream hour',
        'Price': 'Pricing strategy - balance value and accessibility',
        'Average_playtime_forever': 'Game depth - players spend more time on good games',
        'Peak_CCU': 'Peak popularity - maximum concurrent players',
        'Achievements': 'Content richness - more achievements = more content',
        'platform_count': 'Platform accessibility - Windows/Mac/Linux support',
        'is_free': 'Free-to-play model - impacts success differently',
        'Metacritic_score': 'Professional reviews - critic scores',
        'streaming_popularity': 'Overall streaming popularity (calculated)',
        'community_engagement': 'Combined community activity (reviews × viewers)'
    }

    print(f"\nAll {len(feature_imp)} Features by Importance:")
    for i, row in feature_imp.iterrows():
        feature = row['Feature']
        importance = row['Importance']
        explanation = explanations.get(feature, feature)

        if importance > 0.15:
            marker = "🔥"
        elif importance > 0.05:
            marker = "⭐"
        else:
            marker = "  "

        print(f"  {i+1:2d}. {marker} {explanation}")
        print(f"      → Importance: {importance:.1%}")

print("\n" + "="*70)
print("Actionable Insights for Developers")
print("="*70)

# Compare success vs failure
success_games = df_full[df_full['success'] == 1]
failure_games = df_full[df_full['success'] == 0]

print("\nDO:")
print(f"  • Prioritize quality: {success_games['positive_ratio'].mean():.1%} avg positive reviews")
print(f"  • Build community: {success_games['Recommendations'].mean():.0f} avg recommendations")
print(f"  • Price smart: ${success_games['Price'].mean():.2f} avg price")
print(f"  • Create depth: {success_games['Average_playtime_forever'].mean():.0f} min avg playtime")

print("\nAVOID:")
print(f"  • Low quality: <85% positive reviews")
print(f"  • Weak community: <1000 recommendations")
print(f"  • Chasing Twitch views without gameplay quality")

print("\nThe Twitch Paradox:")
print("  High Twitch views ≠ Success")
print("  → Focus on making a great game, not just a watchable one")

# Practical examples
print("\n Prediction Examples:")

high_prob = df_sample.nlargest(1, 'success_probability').iloc[0]
low_prob = df_sample.nsmallest(1, 'success_probability').iloc[0]

print(f"\n   High Success: {high_prob['Name']}")
print(f"     Probability: {high_prob['success_probability']:.1%}")
print(f"     Key: {high_prob['positive_ratio']:.1%} reviews, {high_prob['Recommendations']:.0f} recommendations")

print(f"\n  Low Success: {low_prob['Name']}")
print(f"     Probability: {low_prob['success_probability']:.1%}")
print(f"     Issue: {low_prob['positive_ratio']:.1%} reviews, {low_prob['total_reviews']:.0f} total reviews")

print("\nThank you for watching!")


What Makes a Game Succeed?

All 17 Features by Importance:
   2. 🔥 Player satisfaction - aim for 85%+ positive reviews
      → Importance: 65.3%
   3. 🔥 Player engagement - more reviews = more visibility
      → Importance: 17.9%
   9.    Peak popularity - maximum concurrent players
      → Importance: 2.7%
  10.    Word-of-mouth - players actively recommend your game
      → Importance: 2.1%
   1.    Pricing strategy - balance value and accessibility
      → Importance: 2.1%
   4.    Content richness - more achievements = more content
      → Importance: 1.6%
  12.    Streaming appeal - average viewer count
      → Importance: 1.4%
   8.    Game depth - players spend more time on good games
      → Importance: 1.3%
  17.    Combined community activity (reviews × viewers)
      → Importance: 1.2%
  11.    Marketing reach - Twitch/streaming visibility
      → Importance: 1.0%
  14.    Streamer interest - how many people stream it
      → Importance: 1.0%
  13.    Viral potential - maxi